# Viettel AI Race 2026 — Colab T4 validation

This notebook validates a clean vLLM installation, server stability, the repository's 420-request workload, and accuracy tooling. It is **not** an H200 performance benchmark: never choose a portal submission from T4 TTFT, TPOT, or ERS.

The T4 profile uses FP16 because a Tesla T4 has no native Hopper FP8 W8A8 path. The exact v6 FP8/FP8-KV flags remain available as an opt-in **startup smoke** profile; record whether it starts, but do not compare its latency with H200. `speculative-draft-v6-fp8-smoke` applies those exact target flags with the local draft, while `speculative-draft-smoke` stays FP16 for optional functional triage. Neither profile measures H200 latency.

If an earlier cell imported `torch` or `vllm`, choose **Runtime → Restart session** before running this notebook from the top.

## 1. Clone the repository and install the CUDA 12.9 vLLM wheel

The setup deliberately removes the incompatible PyPI/CUDA-13 installation and any legacy `libcudart.so.13` symlink. It then installs **only** `vllm==0.22.1` through the official `cu129` wheel index and verifies the compiled extension in a fresh subprocess before any model is downloaded.

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
import time
from pathlib import Path

REPO_URL = 'https://github.com/Platypus27-coder/viettel-ai-race-llm-serving.git'
# Set VIETTEL_REPO_REF to a branch, tag, or published commit before this cell
# when an experiment must be replayed exactly. The resolved SHA is always artifacted.
REPO_REF = os.environ.get('VIETTEL_REPO_REF', 'main')
REPO_DIR = Path('/content/viettel-ai-race-llm-serving')
ARTIFACT_ROOT = Path('/content/viettel-artifacts')
RUN_DIR = ARTIFACT_ROOT / f"colab-{time.strftime('%Y%m%d-%H%M%S')}"
MODEL_ID = 'LiquidAI/LFM2.5-1.2B-Instruct'
MODEL_REVISION = os.environ.get('VIETTEL_MODEL_REVISION', 'main')
MODEL_DIR = Path('/content/LFM2.5-1.2B-Instruct')
DRAFT_MODEL_ID = 'LiquidAI/LFM2.5-350M'
# This revision is intentionally immutable: it must match the draft baked into a portal image.
DRAFT_MODEL_REVISION = '1575d1b8b67d862834836087765bff2ef4020672'
DRAFT_MODEL_DIR = Path('/content/LFM2.5-350M')
BASE_URL = 'http://127.0.0.1:8000'
VALID_COLAB_PROFILES = {
    't4-fp16', 'v6-fp8-smoke', 'shortconv-fp8-smoke',
    'speculative-draft-smoke', 'speculative-draft-v6-fp8-smoke',
}
SPECULATIVE_DRAFT_PROFILES = {
    'speculative-draft-smoke', 'speculative-draft-v6-fp8-smoke',
}
# Set VIETTEL_COLAB_PROFILE before the model-download cell when a profile needs extra assets.
COLAB_PROFILE = os.environ.get('VIETTEL_COLAB_PROFILE', 't4-fp16').strip()
if COLAB_PROFILE not in VALID_COLAB_PROFILES:
    raise ValueError('VIETTEL_COLAB_PROFILE must be one of: ' + ', '.join(sorted(VALID_COLAB_PROFILES)))

def run_checked(command: list[str], **kwargs) -> subprocess.CompletedProcess[str]:
    print('$', ' '.join(command))
    kwargs.setdefault('check', True)
    kwargs.setdefault('text', True)
    return subprocess.run(command, **kwargs)

loaded_runtime_modules = sorted({'torch', 'vllm'} & set(sys.modules))
if loaded_runtime_modules:
    raise RuntimeError(
        'This kernel already imported ' + ', '.join(loaded_runtime_modules)
        + '. Select Runtime → Restart session, then run this notebook from the top.'
    )

ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=False)

# A clone/fetch/checkout sequence works on a fresh runtime and can also refresh
# a retained /content directory without relying on an uploaded project folder.
if not REPO_DIR.exists():
    run_checked(['git', 'clone', '--no-checkout', '--depth', '1', REPO_URL, str(REPO_DIR)])
elif not (REPO_DIR / '.git').is_dir():
    raise RuntimeError(f'{REPO_DIR} exists but is not the expected Git clone; remove it and rerun.')

run_checked(['git', '-C', str(REPO_DIR), 'remote', 'set-url', 'origin', REPO_URL])
run_checked(['git', '-C', str(REPO_DIR), 'fetch', '--depth', '1', 'origin', REPO_REF])
run_checked(['git', '-C', str(REPO_DIR), 'checkout', '--detach', '--force', 'FETCH_HEAD'])
REPO_SHA = run_checked(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], capture_output=True).stdout.strip()

required_repo_files = [
    REPO_DIR / 'benchmark' / 'benchmark_ers.py',
    REPO_DIR / 'benchmark' / 'compare_greedy.py',
    REPO_DIR / 'benchmark' / 'test_accuracy.py',
    REPO_DIR / 'docker' / 'shortconv-fp8' / 'patch_vllm_shortconv_fp8.py',
    REPO_DIR / '019e649f-4e27-74db-82da-920f57b13786' / 'grading-workload-spec.json',
    REPO_DIR / 'docker-compose.yml',
]
missing_repo_files = [str(path) for path in required_repo_files if not path.is_file()]
if missing_repo_files:
    raise RuntimeError('Repository checkout is incomplete: ' + ', '.join(missing_repo_files))

# Delete only the broken symlink created by the old notebook. Never fabricate a
# CUDA-13 runtime from a CUDA-12 library.
legacy_cudart_link = Path('/usr/local/lib/libcudart.so.13')
removed_legacy_symlink = False
if legacy_cudart_link.is_symlink():
    legacy_cudart_link.unlink()
    removed_legacy_symlink = True

run_checked([sys.executable, '-m', 'pip', 'uninstall', '-y', 'vllm', 'torch', 'torchvision', 'torchaudio'], check=False)
run_checked([sys.executable, '-m', 'pip', 'cache', 'purge'], check=False)
run_checked([sys.executable, '-m', 'pip', 'install', '--upgrade', 'uv'])

VLLM_WHEEL_INDEX = 'https://wheels.vllm.ai/0.22.1/cu129'
install_command = [
    sys.executable, '-m', 'uv', 'pip', 'install', '--system',
    '--torch-backend=cu129',
    '--extra-index-url', VLLM_WHEEL_INDEX,
    '--index-strategy', 'unsafe-best-match',
    'vllm==0.22.1',
    'aiohttp>=3.9.0', 'openai>=1.0.0', 'numpy>=1.24.0', 'transformers>=4.57.2',
]
run_checked(install_command)

preflight_code = r'''
import json
import torch
import vllm
import vllm._C
payload = {
    'vllm_version': vllm.__version__,
    'compiled_extension_imported': True,
    'torch_version': torch.__version__,
    'torch_cuda': torch.version.cuda,
    'cuda_available': torch.cuda.is_available(),
    'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'gpu_capability': torch.cuda.get_device_capability(0) if torch.cuda.is_available() else None,
}
print('PREFLIGHT_JSON=' + json.dumps(payload, sort_keys=True))
'''
preflight = subprocess.run(
    [sys.executable, '-c', preflight_code], capture_output=True, text=True, check=False
)
(RUN_DIR / 'preflight.stdout.log').write_text(preflight.stdout, encoding='utf-8')
(RUN_DIR / 'preflight.stderr.log').write_text(preflight.stderr, encoding='utf-8')
marker_lines = [line for line in preflight.stdout.splitlines() if line.startswith('PREFLIGHT_JSON=')]
if preflight.returncode != 0 or not marker_lines:
    raise RuntimeError(
        'vLLM CUDA preflight failed before model download.\n'
        + (preflight.stdout + '\n' + preflight.stderr)[-5000:]
    )
environment = json.loads(marker_lines[-1].split('=', 1)[1])
preflight_errors = []
if environment['vllm_version'] != '0.22.1':
    preflight_errors.append(f"Expected vLLM 0.22.1, got {environment['vllm_version']}")
if not environment['compiled_extension_imported']:
    preflight_errors.append('vllm._C did not import')
if not str(environment['torch_cuda']).startswith('12.'):
    preflight_errors.append(f"Expected a CUDA 12 build, got {environment['torch_cuda']}")
if not environment['cuda_available'] or 'T4' not in str(environment['gpu_name']):
    preflight_errors.append(f"Expected a Tesla T4 GPU, got {environment['gpu_name']}")
if preflight_errors:
    raise RuntimeError('Preflight rejected this runtime: ' + '; '.join(preflight_errors))

smi = subprocess.run(['nvidia-smi'], capture_output=True, text=True, check=False)
(RUN_DIR / 'nvidia-smi.txt').write_text(smi.stdout + smi.stderr, encoding='utf-8')
freeze = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True, check=True)
(RUN_DIR / 'pip-freeze.txt').write_text(freeze.stdout, encoding='utf-8')
environment.update({
    'repository_url': REPO_URL,
    'repository_ref_requested': REPO_REF,
    'repository_commit': REPO_SHA,
    'model_id': MODEL_ID,
    'model_revision_requested': MODEL_REVISION,
    'vllm_wheel_index': VLLM_WHEEL_INDEX,
    'removed_legacy_libcudart_symlink': removed_legacy_symlink,
})
(RUN_DIR / 'environment.json').write_text(json.dumps(environment, indent=2), encoding='utf-8')
print(json.dumps(environment, indent=2))
print(f'Artifacts will be written to: {RUN_DIR}')


## 2. Download the target and required pinned draft after the environment preflight passes

This cell is intentionally after the compiled-extension check, so a CUDA mismatch never wastes time downloading model weights. Select a profile before running it: either speculative profile also downloads the immutable LFM2.5-350M draft locally and rejects the run unless its complete tokenizer vocabulary and special-token configuration exactly match the target. The server cell rejects a profile change after this download boundary.

In [ ]:
import hashlib

from huggingface_hub import HfApi, snapshot_download
from transformers import AutoTokenizer

# Permit selecting a draft-dependent profile after setup but before this download cell.
COLAB_PROFILE = os.environ.get('VIETTEL_COLAB_PROFILE', COLAB_PROFILE).strip()
if COLAB_PROFILE not in VALID_COLAB_PROFILES:
    raise ValueError('VIETTEL_COLAB_PROFILE must be one of: ' + ', '.join(sorted(VALID_COLAB_PROFILES)))
PROFILE_AT_DOWNLOAD = COLAB_PROFILE

model_info = HfApi().model_info(MODEL_ID, revision=MODEL_REVISION)
MODEL_RESOLVED_REVISION = model_info.sha
print(f'Downloading {MODEL_ID}@{MODEL_RESOLVED_REVISION} to {MODEL_DIR}')
snapshot_path = snapshot_download(
    repo_id=MODEL_ID,
    revision=MODEL_RESOLVED_REVISION,
    local_dir=str(MODEL_DIR),
)
if not (MODEL_DIR / 'config.json').is_file():
    raise RuntimeError(f'Model download did not create config.json under {MODEL_DIR}')
model_manifest = {
    'model_id': MODEL_ID,
    'revision_requested': MODEL_REVISION,
    'revision_resolved': MODEL_RESOLVED_REVISION,
    'local_path': str(MODEL_DIR),
    'snapshot_path': str(snapshot_path),
}

DRAFT_MODEL_RESOLVED_REVISION = None
tokenizer_validation = None
if COLAB_PROFILE in SPECULATIVE_DRAFT_PROFILES:
    draft_info = HfApi().model_info(DRAFT_MODEL_ID, revision=DRAFT_MODEL_REVISION)
    DRAFT_MODEL_RESOLVED_REVISION = draft_info.sha
    if DRAFT_MODEL_RESOLVED_REVISION != DRAFT_MODEL_REVISION:
        raise RuntimeError(
            'Draft revision was not immutable: expected ' + DRAFT_MODEL_REVISION
            + ', got ' + str(DRAFT_MODEL_RESOLVED_REVISION)
        )
    print(f'Downloading pinned draft {DRAFT_MODEL_ID}@{DRAFT_MODEL_RESOLVED_REVISION} to {DRAFT_MODEL_DIR}')
    draft_snapshot_path = snapshot_download(
        repo_id=DRAFT_MODEL_ID,
        revision=DRAFT_MODEL_RESOLVED_REVISION,
        local_dir=str(DRAFT_MODEL_DIR),
    )
    if not (DRAFT_MODEL_DIR / 'config.json').is_file():
        raise RuntimeError(f'Draft download did not create config.json under {DRAFT_MODEL_DIR}')

    def json_safe(value):
        if isinstance(value, dict):
            return {str(key): json_safe(item) for key, item in value.items()}
        if isinstance(value, (list, tuple)):
            return [json_safe(item) for item in value]
        if value is None or isinstance(value, (str, int, float, bool)):
            return value
        return str(value)

    def tokenizer_summary(tokenizer):
        vocabulary = tokenizer.get_vocab()
        normalized_vocabulary = sorted((str(token), int(token_id)) for token, token_id in vocabulary.items())
        vocabulary_bytes = json.dumps(
            normalized_vocabulary, ensure_ascii=False, separators=(',', ':')
        ).encode('utf-8')
        return vocabulary, {
            'tokenizer_class': tokenizer.__class__.__name__,
            'vocab_size': len(tokenizer),
            'base_vocab_size': tokenizer.vocab_size,
            'vocab_sha256': hashlib.sha256(vocabulary_bytes).hexdigest(),
            'special_tokens_map': json_safe(tokenizer.special_tokens_map),
            'all_special_tokens': [str(token) for token in tokenizer.all_special_tokens],
            'all_special_ids': [int(token_id) for token_id in tokenizer.all_special_ids],
            'bos_token_id': tokenizer.bos_token_id,
            'eos_token_id': tokenizer.eos_token_id,
            'pad_token_id': tokenizer.pad_token_id,
            'unk_token_id': tokenizer.unk_token_id,
        }

    target_tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), trust_remote_code=True)
    draft_tokenizer = AutoTokenizer.from_pretrained(str(DRAFT_MODEL_DIR), trust_remote_code=True)
    target_vocabulary, target_tokenizer_summary = tokenizer_summary(target_tokenizer)
    draft_vocabulary, draft_tokenizer_summary = tokenizer_summary(draft_tokenizer)
    compatibility_errors = []
    if target_vocabulary != draft_vocabulary:
        compatibility_errors.append('complete tokenizer vocabulary/token-ID mapping differs')
    for field in ('special_tokens_map', 'all_special_tokens', 'all_special_ids', 'bos_token_id', 'eos_token_id', 'pad_token_id', 'unk_token_id'):
        if target_tokenizer_summary[field] != draft_tokenizer_summary[field]:
            compatibility_errors.append(f'special-token field differs: {field}')
    tokenizer_validation = {
        'target_model_id': MODEL_ID,
        'target_model_revision_resolved': MODEL_RESOLVED_REVISION,
        'draft_model_id': DRAFT_MODEL_ID,
        'draft_model_revision_pinned': DRAFT_MODEL_REVISION,
        'draft_model_revision_resolved': DRAFT_MODEL_RESOLVED_REVISION,
        'target': target_tokenizer_summary,
        'draft': draft_tokenizer_summary,
        'compatible': not compatibility_errors,
        'errors': compatibility_errors,
    }
    (RUN_DIR / 'tokenizer_compatibility.json').write_text(
        json.dumps(tokenizer_validation, indent=2), encoding='utf-8'
    )
    if compatibility_errors:
        raise RuntimeError(
            'Draft tokenizer is incompatible with target; inspect tokenizer_compatibility.json: '
            + '; '.join(compatibility_errors)
        )
    model_manifest['draft'] = {
        'model_id': DRAFT_MODEL_ID,
        'revision_pinned': DRAFT_MODEL_REVISION,
        'revision_resolved': DRAFT_MODEL_RESOLVED_REVISION,
        'local_path': str(DRAFT_MODEL_DIR),
        'snapshot_path': str(draft_snapshot_path),
        'tokenizer_compatible': True,
    }

model_manifest['profile'] = COLAB_PROFILE
(RUN_DIR / 'model_manifest.json').write_text(json.dumps(model_manifest, indent=2), encoding='utf-8')
environment.update({
    'colab_profile': COLAB_PROFILE,
    'model_revision_resolved': MODEL_RESOLVED_REVISION,
    'draft_model_id': DRAFT_MODEL_ID if DRAFT_MODEL_RESOLVED_REVISION else None,
    'draft_model_revision_pinned': DRAFT_MODEL_REVISION if DRAFT_MODEL_RESOLVED_REVISION else None,
    'draft_model_revision_resolved': DRAFT_MODEL_RESOLVED_REVISION,
})
(RUN_DIR / 'environment.json').write_text(json.dumps(environment, indent=2), encoding='utf-8')
print(json.dumps(model_manifest, indent=2))


## 3. Start a clean server

`t4-fp16` is the default functional/stability profile. Run it first for the unpatched v6 baseline. Set `VIETTEL_COLAB_PROFILE=v6-fp8-smoke` only to test the exact portal v6 FP8 flags, `VIETTEL_COLAB_PROFILE=shortconv-fp8-smoke` to apply the repository's ShortConv patch, `VIETTEL_COLAB_PROFILE=speculative-draft-smoke` for optional FP16 draft triage, or `VIETTEL_COLAB_PROFILE=speculative-draft-v6-fp8-smoke` for the exact portal FP8/FP8-KV target plus local draft. Choose either speculative profile **before the model-download cell**; changing it later is rejected. The ShortConv patch persists in the Colab runtime, so restart from the setup cell before returning to an unpatched profile. Each execution terminates a prior server and creates a separate artifact directory; it does not prewarm a prefix or issue any greedy comparison request.

In [ ]:
import shutil
import urllib.error
import urllib.request

requested_profile = os.environ.get('VIETTEL_COLAB_PROFILE', PROFILE_AT_DOWNLOAD).strip()
if requested_profile != PROFILE_AT_DOWNLOAD or COLAB_PROFILE != PROFILE_AT_DOWNLOAD:
    raise RuntimeError(
        'VIETTEL_COLAB_PROFILE changed after model download. Rerun the model-download cell '
        'so profile-dependent assets and validation are recorded.'
    )
if COLAB_PROFILE not in VALID_COLAB_PROFILES:
    raise ValueError('VIETTEL_COLAB_PROFILE changed after setup; restart from setup to select a valid profile.')
if COLAB_PROFILE in SPECULATIVE_DRAFT_PROFILES:
    if DRAFT_MODEL_RESOLVED_REVISION != DRAFT_MODEL_REVISION or tokenizer_validation is None:
        raise RuntimeError(
            'The speculative draft was not downloaded and validated. Set a speculative-draft '
            "profile, then rerun the model-download cell before starting the server."
        )
    if not tokenizer_validation.get('compatible'):
        raise RuntimeError('Draft tokenizer validation failed; inspect tokenizer_compatibility.json')

if 'server_process' in globals() and server_process.poll() is None:
    print('Stopping the prior vLLM server before starting a clean profile.')
    server_process.terminate()
    try:
        server_process.wait(timeout=30)
    except subprocess.TimeoutExpired:
        server_process.kill()
if 'server_log_handle' in globals() and not server_log_handle.closed:
    server_log_handle.close()

ACTIVE_RUN_DIR = RUN_DIR / f"{COLAB_PROFILE}-{time.strftime('%Y%m%d-%H%M%S')}"
ACTIVE_RUN_DIR.mkdir(parents=True, exist_ok=False)
shutil.copy2(REPO_DIR / 'docker-compose.yml', ACTIVE_RUN_DIR / 'docker-compose.v6-reference.yml')

shortconv_patch_applied = COLAB_PROFILE == 'shortconv-fp8-smoke'
speculative_draft_enabled = COLAB_PROFILE in SPECULATIVE_DRAFT_PROFILES
if shortconv_patch_applied:
    patch_command = [
        sys.executable,
        str(REPO_DIR / 'docker' / 'shortconv-fp8' / 'patch_vllm_shortconv_fp8.py'),
        '--apply', '--verify',
    ]
    patch_result = subprocess.run(patch_command, capture_output=True, text=True, check=False)
    (ACTIVE_RUN_DIR / 'shortconv_patch.log').write_text(
        patch_result.stdout + patch_result.stderr, encoding='utf-8'
    )
    if patch_result.returncode != 0:
        raise RuntimeError('ShortConv patch failed; inspect shortconv_patch.log')


v6_common_args = [
    sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
    f'--model={MODEL_DIR}',
    '--served-model-name=LFM2.5-1.2B-Instruct',
    '--host=0.0.0.0', '--port=8000', '--tensor-parallel-size=1',
    '--max-model-len=8192',
    '--gpu-memory-utilization=0.97',
    '--enable-prefix-caching',
]
speculative_config = None
speculative_portal_target = COLAB_PROFILE == 'speculative-draft-v6-fp8-smoke'
if COLAB_PROFILE in {'v6-fp8-smoke', 'shortconv-fp8-smoke'}:
    profile_args = ['--quantization=fp8', '--kv-cache-dtype=fp8_e4m3']
    profile_note = (
        'ShortConv FP8 patched startup/function/accuracy smoke only.'
        if shortconv_patch_applied else
        'Exact v6 FP8/FP8-KV startup smoke only.'
    ) + ' T4 is not an H200 FP8 latency proxy.'
elif speculative_draft_enabled:
    speculative_config = {
        'method': 'draft_model',
        'model': str(DRAFT_MODEL_DIR),
        'num_speculative_tokens': 4,
        'draft_tensor_parallel_size': 1,
        'max_model_len': 8192,
    }
    if speculative_portal_target:
        profile_args = [
            '--quantization=fp8', '--kv-cache-dtype=fp8_e4m3',
            '--speculative-config=' + json.dumps(speculative_config, separators=(',', ':')),
        ]
        profile_note = (
            'Exact v6 FP8/FP8-KV target plus pinned local draft startup/workload/accuracy smoke. '
            'T4 speculative latency is not an H200 performance signal.'
        )
    else:
        profile_args = [
            '--dtype=float16',
            '--speculative-config=' + json.dumps(speculative_config, separators=(',', ':')),
        ]
        profile_note = (
            'Pinned local draft FP16 startup/workload/accuracy smoke only. '
            'T4 speculative latency is not an H200 performance signal.'
        )
else:
    profile_args = ['--dtype=float16']
    profile_note = (
        'T4 functional, workload, and accuracy profile. Its latency must not select a portal candidate.'
    )

vllm_cmd = [*v6_common_args, *profile_args]
server_env = os.environ.copy()
server_env.update({
    'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',
    'OMP_NUM_THREADS': '1',
    'MKL_NUM_THREADS': '1',
    'VLLM_NO_USAGE_STATS': '1',
    'DO_NOT_TRACK': '1',
    # INFO is used in Colab only so resolved scheduler/Mamba settings are preserved.
    'VLLM_LOGGING_LEVEL': 'INFO',
})
server_config = {
    'profile': COLAB_PROFILE,
    'profile_selected_before_download': PROFILE_AT_DOWNLOAD,
    'profile_note': profile_note,
    'command': vllm_cmd,
    'common_v6_flags': {
        'max_model_len': 8192,
        'gpu_memory_utilization': 0.97,
        'prefix_caching': True,
    },
    'profile_flags': profile_args,
    'shortconv_patch_applied': shortconv_patch_applied,
    'speculative_draft_enabled': speculative_draft_enabled,
    'speculative_portal_target': speculative_portal_target,
    'speculative_config': speculative_config,
    'draft_model': ({
        'model_id': DRAFT_MODEL_ID,
        'revision_pinned': DRAFT_MODEL_REVISION,
        'revision_resolved': DRAFT_MODEL_RESOLVED_REVISION,
        'local_path': str(DRAFT_MODEL_DIR),
        'tokenizer_compatibility_artifact': str(RUN_DIR / 'tokenizer_compatibility.json'),
    } if speculative_draft_enabled else None),
    'repository_commit': REPO_SHA,
    'model_revision_resolved': MODEL_RESOLVED_REVISION,
}
(ACTIVE_RUN_DIR / 'server_config.json').write_text(json.dumps(server_config, indent=2), encoding='utf-8')

server_log_path = ACTIVE_RUN_DIR / 'vllm.log'
server_log_handle = server_log_path.open('w', encoding='utf-8', buffering=1)
print('Starting:', ' '.join(vllm_cmd))
print('WARNING: T4 latency is not an H200 performance signal.')
server_process = subprocess.Popen(
    vllm_cmd, stdout=server_log_handle, stderr=subprocess.STDOUT, env=server_env
)

def capture_resolved_config() -> None:
    server_log_handle.flush()
    log_text = server_log_path.read_text(encoding='utf-8', errors='replace')
    markers = (
        'engine args', 'scheduler', 'max_num_batched_tokens', 'max-num-batched-tokens',
        'max_num_seqs', 'max-num-seqs', 'chunked prefill', 'mamba', 'hybrid',
        'prefix caching', 'kv cache',
    )
    lines = log_text.splitlines()
    selected = [line for line in lines if any(marker in line.lower() for marker in markers)]
    if not selected:
        selected = lines[-200:]
    (ACTIVE_RUN_DIR / 'startup_resolved_config.log').write_text(
        '\n'.join(selected) + ('\n' if selected else ''), encoding='utf-8'
    )

last_health_error = None
for attempt in range(90):
    if server_process.poll() is not None:
        capture_resolved_config()
        raise RuntimeError(server_log_path.read_text(encoding='utf-8', errors='replace')[-6000:])
    try:
        with urllib.request.urlopen(f'{BASE_URL}/health', timeout=3) as response:
            if response.status == 200:
                break
    except (urllib.error.URLError, TimeoutError) as error:
        last_health_error = repr(error)
    time.sleep(2)
else:
    capture_resolved_config()
    raise RuntimeError(f'Server did not become healthy: {last_health_error}')

with urllib.request.urlopen(f'{BASE_URL}/v1/models', timeout=10) as response:
    models_payload = json.load(response)
served_models = [entry.get('id') for entry in models_payload.get('data', [])]
if 'LFM2.5-1.2B-Instruct' not in served_models:
    raise RuntimeError(f'Unexpected served model list: {served_models}')

health_artifact = {
    'health_url': f'{BASE_URL}/health',
    'health_status': 200,
    'served_models': served_models,
    'server_pid': server_process.pid,
}
(ACTIVE_RUN_DIR / 'health.json').write_text(json.dumps(health_artifact, indent=2), encoding='utf-8')
capture_resolved_config()
print('Server healthy. Resolved-config excerpts:', ACTIVE_RUN_DIR / 'startup_resolved_config.log')


## 4. Run the repository workload and quick accuracy check

The benchmark is the repository implementation—no reduced Colab copy. It performs the 70-conversation × six-turn workload (420 requests), saves raw `/metrics` immediately before and after it, and fails its process if any request is unsuccessful or has the wrong output-token count. It sends no greedy-comparison request before this workload. A speculative profile additionally requires the benchmark JSON to show measured, run-scoped acceptance evidence. T4 ERS and latency remain stability artifacts only.

In [ ]:
if server_process.poll() is not None:
    raise RuntimeError('The vLLM server is no longer running; rerun the server cell.')

RUN_FULL_WORKLOAD = True
RUN_QUICK_ACCURACY = True
trace_path = REPO_DIR / '019e649f-4e27-74db-82da-920f57b13786' / 'grading-workload-spec.json'
benchmark_output = ACTIVE_RUN_DIR / 'ers-420.json'

if speculative_draft_enabled and not RUN_FULL_WORKLOAD:
    raise RuntimeError(
        'A speculative profile requires the complete 420-request workload; '
        'otherwise run-scoped acceptance metrics cannot be validated.'
    )

def capture_raw_metrics(stage: str) -> tuple[Path, str]:
    metric_path = ACTIVE_RUN_DIR / f'vllm.metrics.{stage}'
    try:
        with urllib.request.urlopen(f'{BASE_URL}/metrics', timeout=15) as response:
            raw_metrics = response.read().decode('utf-8', errors='replace')
    except Exception as error:
        raw_metrics = f'# metrics unavailable during {stage}: {error!r}\n'
    metric_path.write_text(raw_metrics, encoding='utf-8')
    return metric_path, raw_metrics

# This endpoint read is intentionally the only pre-workload operation: no greedy
# generation or prefix prewarming is allowed before the benchmark begins.
metrics_before_path, raw_metrics_before = capture_raw_metrics('before-workload')
benchmark_result = None

if RUN_FULL_WORKLOAD:
    print('Running 420 requests. T4 latency is not an H200 performance signal.')
    benchmark_command = [
        sys.executable, str(REPO_DIR / 'benchmark' / 'benchmark_ers.py'),
        '--base-url', BASE_URL,
        '--trace', str(trace_path),
        '--tokenizer-path', str(MODEL_DIR),
        '--request-rate', 'inf',
        '--seed', '42',
        '--runs', '1',
        '--output', str(benchmark_output),
    ]
    with (ACTIVE_RUN_DIR / 'benchmark.log').open('w', encoding='utf-8') as benchmark_log:
        benchmark_result = subprocess.run(
            benchmark_command, stdout=benchmark_log, stderr=subprocess.STDOUT, text=True, check=False
        )
metrics_after_path, raw_metrics_after = capture_raw_metrics('after-workload')
# Keep the old filename as a convenience alias for tools that only expect the final snapshot.
(ACTIVE_RUN_DIR / 'vllm.metrics').write_text(raw_metrics_after, encoding='utf-8')

if benchmark_result is not None and benchmark_result.returncode != 0:
    raise RuntimeError(
        'The 420-request workload failed; inspect ' + str(ACTIVE_RUN_DIR / 'benchmark.log')
    )

capture_resolved_config()

speculative_benchmark_gate = {
    'required': speculative_draft_enabled,
    'passed': None,
    'metrics_before_artifact': str(metrics_before_path),
    'metrics_after_artifact': str(metrics_after_path),
}
if speculative_draft_enabled:
    gate_errors = []
    speculative_metrics = None
    drafts_observed = None
    mean_acceptance_length = None
    try:
        benchmark_report = json.loads(benchmark_output.read_text(encoding='utf-8'))
    except Exception as error:
        gate_errors.append(f'could not read benchmark JSON: {error!r}')
        benchmark_report = None
    if benchmark_report is not None:
        benchmark_runs = benchmark_report.get('runs')
        if not isinstance(benchmark_runs, list) or len(benchmark_runs) != 1:
            gate_errors.append('benchmark JSON must contain exactly one completed run')
        else:
            speculative_metrics = benchmark_runs[0].get('speculative_decoding')
            if not isinstance(speculative_metrics, dict):
                gate_errors.append('benchmark JSON has no speculative_decoding result')
            else:
                if speculative_metrics.get('available') is not True:
                    gate_errors.append('speculative metrics are not available')
                if speculative_metrics.get('counter_scope') != 'benchmark_delta':
                    gate_errors.append('speculative metrics are not a benchmark_delta')
                if speculative_metrics.get('acceptance_status') != 'measured':
                    gate_errors.append('speculative acceptance was not measured')
                if speculative_metrics.get('counter_reset_detected') is not False:
                    gate_errors.append('speculative counter reset was detected or not reported')
                counters = speculative_metrics.get('counters')
                if not isinstance(counters, dict):
                    gate_errors.append('speculative counters are missing')
                else:
                    try:
                        drafts_observed = float(counters.get('num_drafts'))
                    except (TypeError, ValueError):
                        gate_errors.append('speculative num_drafts is missing or invalid')
                    else:
                        if drafts_observed <= 0:
                            gate_errors.append('no speculative drafts were observed during the workload')
                try:
                    mean_acceptance_length = float(
                        speculative_metrics.get('mean_acceptance_length')
                    )
                except (TypeError, ValueError):
                    gate_errors.append('mean speculative acceptance length is missing or invalid')
                else:
                    if mean_acceptance_length < 3.5:
                        gate_errors.append(
                            'mean speculative acceptance length is below the required 3.5'
                        )
    speculative_benchmark_gate.update({
        'passed': not gate_errors,
        'minimum_mean_acceptance_length': 3.5,
        'drafts_observed': drafts_observed,
        'mean_acceptance_length': mean_acceptance_length,
        'speculative_decoding': speculative_metrics,
        'errors': gate_errors,
    })
    (ACTIVE_RUN_DIR / 'speculative_benchmark_gate.json').write_text(
        json.dumps(speculative_benchmark_gate, indent=2), encoding='utf-8'
    )
    if gate_errors:
        raise RuntimeError(
            'Speculative benchmark gate failed; inspect speculative_benchmark_gate.json: '
            + '; '.join(gate_errors)
        )

if RUN_QUICK_ACCURACY:
    quick_accuracy_command = [
        sys.executable, str(REPO_DIR / 'benchmark' / 'test_accuracy.py'),
        '--base-url', BASE_URL, '--mode', 'quick',
        '--quick-output', str(ACTIVE_RUN_DIR / 'quick_accuracy.json'),
    ]
    with (ACTIVE_RUN_DIR / 'quick_accuracy.log').open('w', encoding='utf-8') as accuracy_log:
        quick_accuracy_result = subprocess.run(
            quick_accuracy_command, stdout=accuracy_log, stderr=subprocess.STDOUT, text=True, check=False
        )
    if quick_accuracy_result.returncode != 0:
        raise RuntimeError('Quick accuracy check failed; inspect quick_accuracy.log')

print('Workload and quick-accuracy artifacts:', ACTIVE_RUN_DIR)


## 5. Full GPQA gate and artifact download

Full GPQA is mandatory for either speculative profile and defaults to enabled there; retain the opt-in switch only for ordinary baseline smoke tests. The final cell builds a zip containing the repository SHA, vLLM/Torch/CUDA/GPU preflight, server command, startup-resolved configuration, logs, health result, workload JSON, raw before/after metrics, speculative gate evidence when applicable, and GPQA output.

In [ ]:
# A draft candidate cannot become a portal candidate without full GPQA.
RUN_FULL_GPQA = speculative_draft_enabled

if RUN_FULL_GPQA:
    run_checked([
        sys.executable, '-m', 'uv', 'pip', 'install', '--system',
        '--torch-backend=cu129',
        '--index-strategy', 'unsafe-best-match', 'lm-eval[api]>=0.4.9',
    ])
    gpqa_command = [
        sys.executable, str(REPO_DIR / 'benchmark' / 'test_accuracy.py'),
        '--base-url', BASE_URL, '--mode', 'gpqa', '--task', 'gpqa_diamond',
        '--concurrency', '4', '--output', str(ACTIVE_RUN_DIR / 'gpqa_diamond'),
    ]
    with (ACTIVE_RUN_DIR / 'gpqa.log').open('w', encoding='utf-8') as gpqa_log:
        gpqa_result = subprocess.run(
            gpqa_command, stdout=gpqa_log, stderr=subprocess.STDOUT, text=True, check=False
        )
    if gpqa_result.returncode != 0:
        raise RuntimeError('Full GPQA failed; inspect gpqa.log before considering a submission.')
else:
    print('Full GPQA is skipped for this non-speculative smoke run. Set RUN_FULL_GPQA = True to run it.')

capture_resolved_config()
server_log_handle.flush()
run_manifest = {
    'repository_commit': REPO_SHA,
    'profile': COLAB_PROFILE,
    'server_pid': server_process.pid if server_process.poll() is None else None,
    'server_returncode': server_process.poll(),
    'artifact_directory': str(ACTIVE_RUN_DIR),
    'server_config_artifact': str(ACTIVE_RUN_DIR / 'server_config.json'),
    'metrics_artifact': str(ACTIVE_RUN_DIR / 'vllm.metrics'),
    'metrics_before_artifact': str(metrics_before_path),
    'metrics_after_artifact': str(metrics_after_path),
    'speculative_draft': ({
        'enabled': True,
        'config': speculative_config,
        'tokenizer_compatibility_artifact': str(RUN_DIR / 'tokenizer_compatibility.json'),
        'tokenizer_compatible': bool(tokenizer_validation and tokenizer_validation.get('compatible')),
        'benchmark_gate_artifact': str(ACTIVE_RUN_DIR / 'speculative_benchmark_gate.json'),
        'benchmark_gate': speculative_benchmark_gate,
    } if speculative_draft_enabled else {'enabled': False}),
    'note': 'Colab T4 artifacts validate functionality and accuracy only; do not infer H200 latency.',
}
(ACTIVE_RUN_DIR / 'run_manifest.json').write_text(json.dumps(run_manifest, indent=2), encoding='utf-8')
# Archive RUN_DIR rather than only ACTIVE_RUN_DIR so the preflight environment,
# package list, and GPU probe from the setup cell accompany this server run.
archive_path = shutil.make_archive(str(RUN_DIR), 'zip', root_dir=RUN_DIR)
print(f'Artifact archive: {archive_path}')

DOWNLOAD_ARTIFACTS = True
if DOWNLOAD_ARTIFACTS:
    try:
        from google.colab import files
        files.download(archive_path)
    except ImportError:
        print('Not running in Colab; download the archive from the path above.')
